In [0]:
import json
import os

import data.utilities.common as Commonlib
import yaml
from beertech_utils.data.integrity.integrity_check import EmptyDataFrameCheck
from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from pyspark.sql.functions import col

In [0]:
# this entire command block will re-execute each time the "1. Subject Area" drop-down selection is changed - this populates the appropriate configurations in the "2. Configuration File" drop-down
CONFIG_BASE_PATH = "../../configuration/abfss/"

subject_entries = Commonlib.list_directories(CONFIG_BASE_PATH)
dbutils.widgets.dropdown("subject_area", "", [""] + sorted(subject_entries), "1. Subject Area")
subject_area = dbutils.widgets.get("subject_area")

config_selection_path = f"{CONFIG_BASE_PATH}{subject_area}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "2. Configuration File")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}{subject_area}/{config_file}"

config = Commonlib.get_object_config(config_file_path)
source_config = config.get("source")
bronze_config = config.get(ML.bronze)

In [0]:
try:
    medallion = Medallion(config)

    conn_dict = json.loads(dbutils.secrets.get(scope="naz-data-kv", key=config["source"]["container_secret"]))
    client_id = conn_dict["client_id"]
    client_key = conn_dict["client_key"]
    rel_path = config["source"]["path"]
    source_path = conn_dict["hostname"] + rel_path

    container_configs = {
        "fs.azure.account.auth.type": "OAuth",
        "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        "fs.azure.account.oauth2.client.id": dbutils.secrets.get(scope="naz-data-kv", key=client_id),
        "fs.azure.account.oauth2.client.secret": dbutils.secrets.get(scope="naz-data-kv", key=client_key),
        "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/cef04b19-7776-4a94-b89b-375c77a8f936/oauth2/token",
    }

    for key, value in container_configs.items():
        spark.conf.set(key, value)

    medallion.df = spark.read.format(config["source"]["format"]).load(source_path)

    watermark_col = bronze_config.get("watermark", "")
    load_type = bronze_config.get("load_type")

    # incremental load - based on watermark
    if load_type == "append" and watermark_col:
        watermark = medallion.get_watermark(layer=ML.bronze, watermark_col=watermark_col)
        medallion.df = medallion.df.filter(col(watermark_col) > watermark)
        medallion.logger.info(
            f"Performing incremental data load from abfss path based on records where {watermark_col} > {watermark}"
        )
    # full load
    elif load_type == "overwrite":
        empty_df = EmptyDataFrameCheck(dataset_name=medallion.name)
        empty_df.run(medallion.df)
        medallion.logger.info("Performing full data load from abfss path")
    # custom query load
    else:
        medallion.logger.info("Appending records from abfss path based on custom query")
except Exception as e:
    medallion.logger.error(
        f"An error occured while processing the abfss_to_bronze job. subject={subject_area}, config_file={config_file}: {e}"
    )
    raise

In [0]:
try:
    medallion.write_to_delta(
        load_type=load_type,
        layer=ML.bronze,
        source=f"ABFSSPath | {config['source']['path']}",
    )

    medallion.df.unpersist()
except Exception as e:
    medallion.logger.error(
        f"An error occured while writing to bronze via oracle_procedure. subject={subject_area}, config_file={config_file}: {e}"
    )
    raise